# 🎨 Notebook 04 — Customer Segmentation (Clustering)
Methods: K-Means + DBSCAN

Covers: elbow method, silhouette scores, PCA visualization, segment profiling

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data.loader import load_raw_data
from data.preprocessor import ChurnPreprocessor, TARGET, ID_COL
from models.clustering import (
    find_optimal_k, run_kmeans, run_dbscan,
    profile_segments, reduce_to_2d, SEGMENT_NAMES
)

df = load_raw_data()
prep = ChurnPreprocessor(scale=True)
X_proc = prep.fit_transform(df.drop(columns=[TARGET, ID_COL], errors='ignore'))
X = X_proc.values
print(f'Feature matrix: {X.shape}')

## 1. Elbow Method — Find Optimal k

In [ ]:
best_k, inertias, silhouettes, k_range = find_optimal_k(X, range(2, 9))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_range, inertias, 'o-', color='#E63946')
ax1.axvline(best_k, ls='--', color='#F4A261', label=f'Best k={best_k}')
ax1.set(title='Elbow Method — Inertia', xlabel='k', ylabel='Inertia')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(k_range, silhouettes, 'o-', color='#2A9D8F')
ax2.axvline(best_k, ls='--', color='#F4A261', label=f'Best k={best_k}')
ax2.set(title='Silhouette Scores', xlabel='k', ylabel='Silhouette')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'\nOptimal k = {best_k}')

## 2. K-Means Clustering

In [ ]:
km_labels, km_model, km_score = run_kmeans(X, best_k)
coords_2d = reduce_to_2d(X)

df_plot = pd.DataFrame({
    'PC1': coords_2d[:, 0],
    'PC2': coords_2d[:, 1],
    'Segment': pd.Series(km_labels).map(SEGMENT_NAMES).fillna('Other'),
    'Churn': df[TARGET].values
})

colors = ['#2A9D8F', '#457B9D', '#E63946', '#F4A261']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i, seg in enumerate(df_plot['Segment'].unique()):
    mask = df_plot['Segment'] == seg
    ax1.scatter(df_plot.loc[mask, 'PC1'], df_plot.loc[mask, 'PC2'],
                c=colors[i % len(colors)], label=seg, alpha=0.5, s=8)
ax1.set(title='K-Means Clusters (PCA 2D)', xlabel='PC1', ylabel='PC2')
ax1.legend(markerscale=3, fontsize=8); ax1.grid(alpha=0.2)

for churn_val, marker, label in [(0, 'o', 'Retained'), (1, 'x', 'Churned')]:
    mask = df_plot['Churn'] == churn_val
    ax2.scatter(df_plot.loc[mask, 'PC1'], df_plot.loc[mask, 'PC2'],
                marker=marker, alpha=0.3, s=8,
                c='#2A9D8F' if churn_val == 0 else '#E63946', label=label)
ax2.set(title='Churn Labels (PCA 2D)', xlabel='PC1', ylabel='PC2')
ax2.legend(markerscale=3); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3. DBSCAN

In [ ]:
db_labels, db_model = run_dbscan(X)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()

print(f'DBSCAN Clusters: {n_clusters}')
print(f'Noise / Outlier points: {n_noise} ({n_noise/len(db_labels):.1%})')

fig, ax = plt.subplots(figsize=(7, 5))
unique_labels = sorted(set(db_labels))
palette = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 1)))
for label, color in zip(unique_labels, palette):
    mask = db_labels == label
    lbl  = 'Noise' if label == -1 else f'Cluster {label}'
    ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1],
               c=[color], label=lbl, alpha=0.5, s=8,
               marker='x' if label == -1 else 'o')
ax.set(title='DBSCAN Clusters (PCA 2D)', xlabel='PC1', ylabel='PC2')
ax.legend(markerscale=3, fontsize=8); ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Segment Profiling

In [ ]:
profile = profile_segments(df, km_labels)
profile.reset_index(inplace=True)

# Churn rate by segment
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_to_plot = [
    ('churn_rate' if 'churn_rate' in profile.columns else 'count', 'Churn Rate', '#E63946'),
    ('monthly_charges' if 'monthly_charges' in profile.columns else 'count', 'Avg Monthly Charges', '#457B9D'),
    ('tenure' if 'tenure' in profile.columns else 'count', 'Avg Tenure (months)', '#2A9D8F'),
]
for ax, (col, title, color) in zip(axes, metrics_to_plot):
    if col in profile.columns:
        bars = ax.bar(profile.get('segment_name', profile['segment']), profile[col], color=color)
        ax.set_title(title, fontweight='bold')
        ax.tick_params(axis='x', rotation=20)
        ax.grid(axis='y', alpha=0.3)
        if 'rate' in col:
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.suptitle('Segment Profiles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()